# Launch Standalone SWC Box Selector

This launches the desktop PyVista selector outside Jupyter widgets.

Window controls:
- `r` then drag: box-select compartments (additive)
- `c`: clear selection
- `g`: apply glia-loss to selected
- `s`: save selected spec JSON
- `p`: print selected list


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys


def _find_work_root() -> Path:
    candidates = []
    env_root = os.environ.get('DIGIFLY_VIP_GLIA_ROOT', '').strip()
    if env_root:
        candidates.append(Path(env_root))
    try:
        candidates.append(Path.cwd())
    except Exception:
        pass
    try:
        cwd = Path.cwd()
        for base in [cwd] + list(cwd.parents):
            candidates.append(base / 'Phase 2' / 'apps' / 'VIP_Glia_Sim')
            candidates.append(base / 'apps' / 'VIP_Glia_Sim')
    except Exception:
        pass
    for candidate in candidates:
        try:
            candidate = candidate.expanduser().resolve()
        except Exception:
            continue
        for root in [candidate] + list(candidate.parents):
            if (root / 'tools' / 'swc_box_selector_app.py').exists():
                return root
    raise RuntimeError('Could not locate VIP_Glia_Sim app root. Set DIGIFLY_VIP_GLIA_ROOT.')


WORK_ROOT = _find_work_root()
PHASE2_ROOT = Path(os.environ.get('DIGIFLY_PHASE2_ROOT', str(WORK_ROOT.parents[1]))).expanduser().resolve()
APP_PATH = WORK_ROOT / 'tools' / 'swc_box_selector_app.py'
SWC_DIR = Path(os.environ.get('DIGIFLY_SWC_DIR', str(PHASE2_ROOT / 'data' / 'export_swc'))).expanduser().resolve()

NEURON_ID = 10068
KO_MM = 6.5
KI_MM = 140.0
POINT_STRIDE = 4
SELECTED_COLOR = '#ff0000'
OUT_JSON = WORK_ROOT / 'notebooks' / 'debug' / 'outputs' / 'selected_glia_spec.json'

print('APP_PATH   =', APP_PATH)
print('SWC_DIR    =', SWC_DIR)
print('PHASE2_ROOT=', PHASE2_ROOT)
print('OUT_JSON   =', OUT_JSON)


In [ ]:
cmd = [
    sys.executable,
    str(APP_PATH),
    '--neuron-id', str(NEURON_ID),
    '--swc-dir', str(SWC_DIR),
    '--phase2-root', str(PHASE2_ROOT),
    '--ko-mM', str(KO_MM),
    '--ki-mM', str(KI_MM),
    '--point-stride', str(POINT_STRIDE),
    '--selected-color', str(SELECTED_COLOR),
    '--output-spec-json', str(OUT_JSON),
]
print(shlex.join(cmd))


In [ ]:
env = os.environ.copy()
selector_proc = subprocess.Popen(cmd, cwd=str(WORK_ROOT), env=env)
print(f'Launched selector (PID={selector_proc.pid}). Close the desktop window when done.')


In [ ]:
# Optional: stop a still-running selector process.
if 'selector_proc' in globals() and selector_proc.poll() is None:
    selector_proc.terminate()
    print('Sent terminate to PID', selector_proc.pid)
else:
    print('No running selector process found in this kernel.')
